In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')

#Load & Inspect Dataset

In [ ]:
df = pd.read_csv('Sample-Superstore.csv', encoding='latin-1')

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.info()

In [ ]:
df.describe()

#Numerical or Categorical Columns ko alag karo

In [ ]:
num_col = df.select_dtypes(include=['number']).columns.tolist()
print(num_col)

In [ ]:
cat_col = df.select_dtypes(include=['object', 'category']).columns.tolist()
cat_col

In [ ]:
df.nunique().sort_values()

#Missing Values Analyze

In [ ]:
missing = df.isnull().sum()
missing

#Duplicate Detection

In [ ]:
duplicate = df.duplicated().sum()
duplicate

#Sekwenss Check karo

In [ ]:
df[num_col].describe().round(2)

In [ ]:
df[num_col].skew().round(3)

In [ ]:
def plotting(var, num):
    plt.subplot(2, 2, num)
    sns.histplot(df[var], kde=True)
    skew_val = df[var].skew()
    plt.title(f'{var}\nSkewness: {skew_val:.3f}')

# Call the function
plotting('Sales', 1)
plotting('Discount', 2)
plotting('Quantity', 3)
plotting('Profit', 4)

plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import yeojohnson
df['Sales'] = np.log1p(df['Sales'])
df['Profit'], _ = yeojohnson(df['Profit'] - df['Profit'].min() + 0.01)
df['Quantity'], _ = yeojohnson(df['Quantity'] + 0.01)
df['Discount'], _ = yeojohnson(df['Discount'] + 0.01)

# ✅ IMPORTANT: Convert to float (yeojohnson kabhi kabhi int return kar deta hai)
df['Profit'] = df['Profit'].astype(float)
df['Quantity'] = df['Quantity'].astype(float)
df['Discount'] = df['Discount'].astype(float)

columns = ['Sales', 'Profit', 'Quantity', 'Discount']
df[columns].skew().round(3)

#Outliers detect karo or un ko solve karo

In [ ]:
# STEP 1: Function define karo
def remove_outliers_iqr(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    df_clean = df[(df[column] >= lower) & (df[column] <= upper)]
    print(f"{column}: Removed {len(df) - len(df_clean)} outliers")
    return df_clean

# STEP 2: Ab outliers remove karo
df = remove_outliers_iqr(df, 'Profit')
df = remove_outliers_iqr(df, 'Sales')

In [ ]:
columns = ['Sales', 'Profit', 'Quantity', 'Discount']
df[columns].skew().round(3)

In [ ]:
df.shape

#Ab numerical colum ka co-relation check karo 

In [ ]:
plt.figure(figsize=(12,6))
sns.heatmap(df[num_col].corr(numeric_only=True), annot=True , linewidths=0.5, )

In [ ]:
df = df.drop(columns=[ 'Row ID','Postal Code','Order ID','Customer ID','Customer Name','Product ID','Product Name','Country','City','State','Sub-Category' ])

In [ ]:
df.nunique().sort_values()

In [ ]:
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date'] = pd.to_datetime(df['Ship Date'])

# Sirf ye 3 features banao
df['Order_Month'] = df['Order Date'].dt.month
df['Order_Quarter'] = df['Order Date'].dt.quarter
df['Shipping_Delay'] = (df['Ship Date'] - df['Order Date']).dt.days

# Old date columns drop karo
df = df.drop(columns=['Order Date', 'Ship Date'])

In [ ]:
df.head()

In [ ]:
df_final = df.copy()

#Feature Engineering

In [ ]:
X = df_final.drop(columns=['Sales'])
y = df_final['Sales']

In [ ]:
X.shape

In [ ]:
X_encode = pd.get_dummies(X, drop_first=True)

In [ ]:
X_encode.head()

In [ ]:
X_encode.astype(int)

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

In [ ]:
# numerical_cols = ['Quantity', 'Discount', 'Profit', 'Order_Month', 'Order_Quarter', 'Shipping_Delay']
X_encode_scaled = scaler.fit_transform(X_encode)

In [ ]:
X_encode_scaled

In [ ]:
# X_final = np.hstack([X_encode,X_encode_scaled])

In [ ]:
X_encode_scaled.shape

In [ ]:
feature_columns = X_encode.columns.tolist()

In [ ]:
X_final = X_encode_scaled

In [ ]:
X_final.shape

In [ ]:
#Model Training

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_final, y, test_size=0.33, random_state=42)

In [ ]:
from sklearn.linear_model import LinearRegression
model = LinearRegression()

In [ ]:
model.fit(X_train,y_train)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

In [ ]:
r2 = r2_score(y_test,y_pred)
r2

In [ ]:
Rmse = mean_squared_error(y_test,y_pred)
Rmse

In [ ]:
from sklearn.ensemble import RandomForestRegressor

In [ ]:
rf_model = RandomForestRegressor(
    n_estimators=80,
    max_depth=10,        # default None hai (unlimited) — limit karo
    min_samples_split=8, # default 2 hai — increase karo
    min_samples_leaf=4,   # default 1 hai — increase karo
    max_features='sqrt',  # already default, but explicitly set karo
    random_state=42
)
rf_model.fit(X_train, y_train)

In [ ]:
y_pred_rf = rf_model.predict(X_test)

In [ ]:
r2_rf = r2_score(y_test,y_pred_rf)
r2_rf

In [ ]:
Rmse_rf = mean_squared_error(y_test,y_pred_rf)
Rmse_rf

In [ ]:
# Train score bhi nikalo
r2_train = r2_score(y_train, rf_model.predict(X_train))
r2_test = r2_score(y_test, rf_model.predict(X_test))

print(f"Train R²: {r2_train:.4f}")
print(f"Test R²:  {r2_test:.4f}")
print(f"Difference: {r2_train - r2_test:.4f}")

In [ ]:
import joblib
joblib.dump(rf_model, 'Random_Forest_model.pkl')
joblib.dump(scaler, 'Scaler.pkl')
joblib.dump(feature_columns, 'feature_columns.pkl')